In [1]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.model_selection import ParameterSampler

try:
    from statsmodels.tsa.arima.model import ARIMA
except ImportError as exc:
    raise ImportError(
        "statsmodels is required. Run this in a notebook cell first: %pip install statsmodels"
    ) from exc

warnings.filterwarnings("ignore")


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

In [3]:
def make_features(series):
    df = pd.DataFrame({"y": series})

    # Lag variables: previous month, 2 months ago, etc.
    df["lag_1"] = df["y"].shift(1)
    df["lag_2"] = df["y"].shift(2)
    df["lag_3"] = df["y"].shift(3)
    df["lag_6"] = df["y"].shift(6)
    df["lag_12"] = df["y"].shift(12)   # same month last year

    # Rolling variables: averages over previous months
    # Important: shift(1) prevents using the current month's sales to predict itself
    df["rolling_mean_3"] = df["y"].shift(1).rolling(window=3).mean()
    df["rolling_mean_6"] = df["y"].shift(1).rolling(window=6).mean()
    df["rolling_mean_12"] = df["y"].shift(1).rolling(window=12).mean()

    df["rolling_std_3"] = df["y"].shift(1).rolling(window=3).std()
    df["rolling_std_6"] = df["y"].shift(1).rolling(window=6).std()
    df["rolling_std_12"] = df["y"].shift(1).rolling(window=12).std()

    # Optional calendar features
    df["month"] = df.index.month
    df["year"] = df.index.year

    return df.dropna()


In [3]:
# Load data and reshape monthly columns into one time series
raw_df = pd.read_csv("Chevy_US_Data.csv")

month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_to_num = {month: i + 1 for i, month in enumerate(month_cols)}

sales_long = (
    raw_df[["Year", *month_cols]]
    .melt(id_vars="Year", value_vars=month_cols, var_name="Month", value_name="Sales")
)

sales_long["MonthNum"] = sales_long["Month"].map(month_to_num)
sales_long["Date"] = pd.to_datetime(
    dict(year=sales_long["Year"], month=sales_long["MonthNum"], day=1)
)

sales_ts = (
    sales_long.sort_values("Date")
    .set_index("Date")["Sales"]
    .astype(float)
    .asfreq("MS")
)

print(f"Observations: {len(sales_ts)}")
print(f"Date range: {sales_ts.index.min().date()} -> {sales_ts.index.max().date()}")

plt.figure(figsize=(12, 4))
plt.plot(sales_ts.index, sales_ts)
plt.title("Chevy Monthly US Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Chronological split (no shuffling for time series)
val_horizon = 12
test_horizon = 12

train = sales_ts.iloc[: -(val_horizon + test_horizon)]
val = sales_ts.iloc[-(val_horizon + test_horizon): -test_horizon]
test = sales_ts.iloc[-test_horizon:]

print(f"Train months: {len(train)} | {train.index.min().date()} -> {train.index.max().date()}")
print(f"Val months:   {len(val)} | {val.index.min().date()} -> {val.index.max().date()}")
print(f"Test months:  {len(test)} | {test.index.min().date()} -> {test.index.max().date()}")

# Small ARIMA grid search using validation RMSE
orders = list(itertools.product(range(0, 4), range(0, 3), range(0, 4)))

best_order = None
best_rmse = np.inf
order_scores = []

for order in orders:
    try:
        fitted = ARIMA(train, order=order).fit()
        val_forecast = fitted.forecast(steps=len(val))
        rmse_score = rmse(val.to_numpy(), val_forecast.to_numpy())
        order_scores.append((order, rmse_score))

        if rmse_score < best_rmse:
            best_rmse = rmse_score
            best_order = order
    except Exception:
        continue

if best_order is None:
    raise RuntimeError("No ARIMA model converged during grid search.")

print(f"Best ARIMA order: {best_order}")
print(f"Validation RMSE: {best_rmse:,.2f}")

print("\nTop 5 orders by validation RMSE:")
for order, score in sorted(order_scores, key=lambda x: x[1])[:5]:
    print(f"order={order} | RMSE={score:,.2f}")